### Hybrid Search Langchain 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("PINECONE_DB")

In [5]:
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone, ServerlessSpec
index_name = "hybrid-search-langchain-pinecone"

# initialize the Pinecone client

pc = Pinecone(api_key=api_key)

# Create the index

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension= 384,
        metric = "dotproduct",
        spec = ServerlessSpec(cloud='aws',region='us-east-1')
    )

In [6]:
index = pc.Index(index_name)
index

/home/common/gen-ai/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 307.17it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default()
bm25_encoder

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/naveenacad89/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/naveenacad89/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [12]:
sentences = [
    "In 2023, I visisted paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
]

## tfidf values on these sentences
bm25_encoder.fit(sentences)


bm25_encoder.dump("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00, 2489.69it/s]


In [13]:
retriever = PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)

In [15]:
retriever


PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x76ab028176d0>, index=<pinecone.db_data.index.Index object at 0x76aca272b490>)

In [17]:
retriever.add_texts(
     [  "In 2023, I visisted paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"]
)

100%|██████████| 1/1 [00:01<00:00,  1.94s/it]


In [19]:
retriever.invoke("What  did i se last")

[Document(metadata={'score': 0.093503}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.0648708344}, page_content='In 2023, I visisted paris'),
 Document(metadata={'score': 0.109449387}, page_content='In 2022, I visited New York')]